## VIATICOS CONSOLIDADO

In [0]:
%pip install openpyxl xlrd 

import pandas as pd
import os
import glob
import re

# ==========================================
# CONFIGURACIÓN PARA DATABRICKS
# ==========================================
# Usamos la ruta de tu repositorio en el Workspace
RUTA_REPO = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC"
RUTA_BASE = f"{RUTA_REPO}/data_corpac/viaticos"
RUTA_SALIDA = f"{RUTA_REPO}/data_corpac/viaticos_consolidados.csv"

def normalizar_rubro(texto):
    if pd.isna(texto) or str(texto).strip() == "": return "OTRO"
    t = str(texto).strip().lower()
    
    if 'viát' in t or 'viat' in t: return 'VIATICOS'
    if 'pasaje' in t: return 'PASAJES'
    if 'telefon' in t or 'celular' in t or 'comunic' in t: return 'TELEFONIA'
    if 'vehíc' in t or 'vehic' in t or 'combust' in t: return 'VEHICULOS'
    if 'publicidad' in t: return 'PUBLICIDAD'
    
    return 'OTRO'

def obtener_trimestre(nombre_archivo):
    n = str(nombre_archivo).upper()
    if "CIERRE" in n or "ANUAL" in n: return None
    if any(x in n for x in ["IVTRIM", "IV_TRIM", "4TRIM"]): return 4
    if any(x in n for x in ["IIITRIM", "III_TRIM", "3TRIM"]): return 3
    if any(x in n for x in ["IITRIM", "II_TRIM", "2TRIM"]): return 2
    if any(x in n for x in ["ITRIM", "I_TRIM", "1TRIM"]): return 1
    return None

def transformar_data(ruta_excel, anio, trimestre):
    try:
        xl = pd.ExcelFile(ruta_excel)
        hoja_objetivo = None
        for sheet in xl.sheet_names:
            if any(x in sheet.upper() for x in ["F9", "FORMATO 9", "VIAJES"]):
                hoja_objetivo = sheet
                break
        
        df_raw = pd.read_excel(xl, sheet_name=hoja_objetivo if hoja_objetivo else 0, header=None)
        datos_extraidos = []

        for r_idx, fila in df_raw.iterrows():
            for c_idx, celda in enumerate(fila):
                rubro_limpio = normalizar_rubro(celda)
                if rubro_limpio != "OTRO":
                    monto_encontrado = None
                    
                    # Caso A: Monto ABAJO
                    if r_idx + 1 < len(df_raw):
                        val_abajo = df_raw.iloc[r_idx + 1, c_idx]
                        monto_ab = pd.to_numeric(str(val_abajo).replace(',', ''), errors='coerce')
                        if pd.notna(monto_ab) and monto_ab != float(anio):
                            monto_encontrado = monto_ab

                    # Caso B: Monto DERECHA
                    if monto_encontrado is None:
                        fila_restante = fila.tolist()[c_idx + 1:]
                        for val_der in reversed(fila_restante):
                            m_der = pd.to_numeric(str(val_der).replace(',', ''), errors='coerce')
                            if pd.notna(m_der) and m_der != float(anio):
                                monto_encontrado = m_der
                                break
                    
                    if monto_encontrado is not None:
                        datos_extraidos.append({'RUBRO': rubro_limpio, 'MONTO': monto_encontrado})

        if not datos_extraidos: return None
        return pd.DataFrame(datos_extraidos).assign(ANIO=anio, TRIMESTRE=trimestre)

    except Exception as e:
        print(f"⚠️ Error en {os.path.basename(ruta_excel)}: {e}")
        return None

# ==========================================
# PROCESO PRINCIPAL
# ==========================================
listado_final = []
print(f"🚀 Iniciando transformación en Databricks...")

if os.path.exists(RUTA_BASE):
    # En Databricks/Linux listdir funciona igual
    carpetas = [d for d in os.listdir(RUTA_BASE) if d.isdigit()]
    for anio_f in sorted(carpetas):
        ruta_anio = os.path.join(RUTA_BASE, anio_f)
        archivos = glob.glob(os.path.join(ruta_anio, "*.[xX][lL][sS]*"))
        
        for r_excel in archivos:
            nom_arch = os.path.basename(r_excel)
            if "~$" in nom_arch: continue
            
            trim = obtener_trimestre(nom_arch)
            if trim:
                df_t = transformar_data(r_excel, int(anio_f), trim)
                if df_t is not None:
                    listado_final.append(df_t)
                    print(f"✅ Procesado: {anio_f} - T{trim}")

if listado_final:
    full_df = pd.concat(listado_final, ignore_index=True)
    full_df = full_df.groupby(['ANIO', 'TRIMESTRE', 'RUBRO'])['MONTO'].sum().reset_index()
    
    # Guardar resultado en el repo
    full_df.to_csv(RUTA_SALIDA, index=False, encoding='utf-8-sig')
    print(f"\n✨ ¡ÉXITO! CSV generado en: {RUTA_SALIDA}")
    
    # Visualización profesional en el Notebook
    display(full_df)
else:
    print("\n❌ No se encontraron datos. Verifica que la ruta base sea correcta.")

### Estructura de BD



In [0]:
%sql
-- Crear la base de datos o esquema si no existe
CREATE SCHEMA IF NOT EXISTS finanzas_viaticos;
USE finanzas_viaticos;

-- 1. Dimensión de Rubros
CREATE TABLE dim_rubro (
    id_rubro BIGINT GENERATED ALWAYS AS IDENTITY,
    nombre_rubro STRING NOT NULL
) USING DELTA;

-- 2. Dimensión de Tiempo
CREATE TABLE dim_tiempo (
    id_tiempo BIGINT GENERATED ALWAYS AS IDENTITY,
    anio INT NOT NULL,
    trimestre INT NOT NULL
) USING DELTA;

-- 3. Tabla de Hechos
CREATE TABLE fact_viaticos (
    id_viatico BIGINT GENERATED ALWAYS AS IDENTITY,
    id_tiempo BIGINT,
    id_rubro BIGINT,
    monto DOUBLE
) USING DELTA;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# 1. Definimos la ruta tal cual aparece en Databricks
# Eliminamos el prefijo 'file:' y usamos la ruta absoluta del sistema
path_workspace = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC/data_corpac/viaticos_consolidados.csv"

# Verificación de seguridad: Esto imprimirá si Databricks realmente ve el archivo
try:
    dbutils.fs.ls(f"file:{path_workspace}")
    print("Archivo encontrado correctamente.")
except:
    print("No se encontró el archivo. Por favor, verifica que la ruta sea exacta (mayúsculas/minúsculas).")

# 2. Carga del DataFrame
# Usamos el formato directo. Si falla 'file:', Databricks a veces prefiere la ruta sin prefijo 
# o movida a una ubicación temporal.
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"file:{path_workspace}")

# --- PROCESAMIENTO (Igual que el anterior) ---

# Cargar Dim_Rubro
df_rubros_unicos = df_raw.select("RUBRO").distinct().withColumnRenamed("RUBRO", "nombre_rubro")
df_rubros_unicos.write.format("delta").mode("append").saveAsTable("finanzas_viaticos.dim_rubro")

# Cargar Dim_Tiempo
df_tiempo_unico = df_raw.select("ANIO", "TRIMESTRE").distinct() \
    .withColumnRenamed("ANIO", "anio") \
    .withColumnRenamed("TRIMESTRE", "trimestre")
df_tiempo_unico.write.format("delta").mode("append").saveAsTable("finanzas_viaticos.dim_tiempo")

# Mapear IDs y cargar Fact_Viaticos
dim_rubro_cargada = spark.table("finanzas_viaticos.dim_rubro")
dim_tiempo_cargada = spark.table("finanzas_viaticos.dim_tiempo")

df_fact_final = df_raw.alias("csv") \
    .join(dim_rubro_cargada.alias("r"), col("csv.RUBRO") == col("r.nombre_rubro")) \
    .join(dim_tiempo_cargada.alias("t"), 
          (col("csv.ANIO") == col("t.anio")) & (col("csv.TRIMESTRE") == col("t.trimestre"))) \
    .select(
        col("t.id_tiempo"),
        col("r.id_rubro"),
        col("csv.MONTO").alias("monto")
    )

df_fact_final.write.format("delta").mode("append").saveAsTable("finanzas_viaticos.fact_viaticos")

print("¡Carga finalizada con éxito!")

## PENALIDADES CONSOLIDADO

In [0]:
# MAGIC %pip install openpyxl xlrd

import pandas as pd
import os
import glob
import re

# ==========================================
# CONFIGURACIÓN PARA DATABRICKS
# ==========================================
RUTA_REPO = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC"
RUTA_BASE = f"{RUTA_REPO}/data_corpac/penalidades"
RUTA_SALIDA = f"{RUTA_REPO}/data_corpac/penalidades_consolidadas.csv"

def extraer_periodo(nombre_archivo):
    n = nombre_archivo.upper()
    anio_match = re.search(r'20\d{2}', n)
    anio = anio_match.group(0) if anio_match else "S/A"
    
    meses = {
        'ENERO': '01', 'FEBRERO': '02', 'MARZO': '03', 'ABRIL': '04', 'MAYO': '05', 'JUNIO': '06',
        'JULIO': '07', 'AGOSTO': '08', 'SETIEMBRE': '09', 'OCTUBRE': '10', 'NOVIEMBRE': '11', 'DICIEMBRE': '12'
    }
    for mes_nom, mes_num in meses.items():
        if mes_nom in n: return anio, f"MES {mes_num}"

    if any(x in n for x in ["IVTRIM", "4TRIM"]): return anio, "TRIM 4"
    if any(x in n for x in ["IIITRIM", "3TRIM"]): return anio, "TRIM 3"
    if any(x in n for x in ["IITRIM", "2TRIM"]): return anio, "TRIM 2"
    if any(x in n for x in ["ITRIM", "1TRIM"]): return anio, "TRIM 1"
    return anio, "S/P"

def clean_currency(x):
    if pd.isna(x) or str(x).strip() == '': return 0.0
    if isinstance(x, (int, float)): return float(x)
    s = re.sub(r'[^\d.]', '', str(x).replace(',', '.'))
    try:
        return float(s)
    except:
        return 0.0

def clean_ruc(x):
    if pd.isna(x): return ""
    s = str(x).strip()
    if s.endswith('.0'): s = s[:-2] # Quita el .0 que pone Excel
    return s

def procesar_excel_penalidades(ruta, anio, periodo):
    try:
        xl = pd.ExcelFile(ruta)
        hoja = next((s for s in xl.sheet_names if any(x in s.upper() for x in ["F7", "FORMATO 7", "PENALIDAD"])), xl.sheet_names[0])
        
        df_temp = pd.read_excel(ruta, sheet_name=hoja, nrows=25, header=None)
        
        fila_cabecera = None
        for i, row in df_temp.iterrows():
            valores_fila = [str(x).upper() for x in row.fillna('').values]
            if any("PROVEEDOR" in s for s in valores_fila) and any("PENALIDAD" in s for s in valores_fila):
                fila_cabecera = i
                break
        
        if fila_cabecera is None: return None 

        df = pd.read_excel(ruta, sheet_name=hoja, skiprows=fila_cabecera)
        df.columns = [re.sub(r'\s+', ' ', str(c)).strip().upper() for c in df.columns]

        # Manejo de Duplicados
        cols = pd.Series(df.columns)
        for dup in cols[cols.duplicated()].unique():
            cols[cols == dup] = [f"{dup}_{i}" if i != 0 else dup for i in range(sum(cols == dup))]
        df.columns = cols

        mapeo_posibilidades = {
            'PROVEEDOR': ['NOMBRE DEL PROVEEDOR O CONTRATISTA', 'PROVEEDOR/CONTRATISTA', 'NOMBRE DEL PROVEEDOR', 'PROVEEDOR'],
            'RUC': ['RUC DEL PROVEEDOR O CONTRATISTA', 'RUC'],
            'MONTO_PENALIDAD': ['MONTO DE LA PENALIDAD S/.', 'MONTO DE LA PENALIDAD', 'MONTO DE LA PENALIDAD S/', 'MONTO DE LA PENALIDAD S/. 1/'],
            'MONTO_TOTAL_CONTRATO': ['MONTO TOTAL DEL CONTRATO S/.', 'MONTO TOTAL DEL CONTRATO S/', 'MONTO TOTAL DEL CONTRATO']
        }

        dict_renombrar = {}
        for destino, origenes in mapeo_posibilidades.items():
            for orig in origenes:
                col_encontrada = next((c for c in df.columns if c == orig or c.startswith(f"{orig}_")), None)
                if col_encontrada:
                    dict_renombrar[col_encontrada] = destino
                    break
        
        df = df.rename(columns=dict_renombrar)

        # Limpieza de montos
        for col in ['MONTO_PENALIDAD', 'MONTO_TOTAL_CONTRATO']:
            if col in df.columns:
                if isinstance(df[col], pd.DataFrame): df[col] = df[col].iloc[:, 0] 
                df[col] = df[col].apply(clean_currency)
        
        # LIMPIEZA DE RUC (CRÍTICO PARA EL ERROR)
        if 'RUC' in df.columns:
            df['RUC'] = df['RUC'].apply(clean_ruc)

        df['ANIO'] = str(anio)
        df['PERIODO'] = periodo
        df['ARCHIVO'] = os.path.basename(ruta)
        
        return df
    except Exception as e:
        print(f"⚠️ Error en {os.path.basename(ruta)}: {e}")
        return None

# ==========================================
# PROCESO PRINCIPAL
# ==========================================
print(f"🚀 Iniciando consolidación de penalidades...")
listado_dfs = []

if os.path.exists(RUTA_BASE):
    archivos = glob.glob(os.path.join(RUTA_BASE, "**/*.xls*"), recursive=True)
    
    for f in archivos:
        if "~$" in f: continue
        nom = os.path.basename(f)
        anio_real, periodo = extraer_periodo(nom)
        
        if anio_real == "S/A":
            partes = f.split(os.sep)
            for p in partes:
                if p.isdigit() and len(p) == 4:
                    anio_real = p
                    break

        df_res = procesar_excel_penalidades(f, anio_real, periodo)
        if df_res is not None:
            listado_dfs.append(df_res)
            print(f"✅ Procesado: {nom}")

if listado_dfs:
    df_final = pd.concat(listado_dfs, ignore_index=True, sort=False)
    
    # Filtrado
    df_final = df_final[
        (df_final['PROVEEDOR'].notna()) & 
        (df_final['PROVEEDOR'].astype(str).str.strip() != '') & 
        (df_final['MONTO_PENALIDAD'] > 0)
    ].copy()

    # Asegurar que RUC sea string en todo el DF final
    df_final['RUC'] = df_final['RUC'].astype(str)

    # Ordenamiento
    df_final['_ORDEN_PERIODO'] = df_final['PERIODO'].replace({'S/P': '99'})
    df_final = df_final.sort_values(by=['ANIO', '_ORDEN_PERIODO'], ascending=[True, True])

    cols_finales = ['ANIO', 'PERIODO', 'PROVEEDOR', 'RUC', 'MONTO_PENALIDAD', 'MONTO_TOTAL_CONTRATO', 'ARCHIVO']
    
    for col in cols_finales:
        if col not in df_final.columns: df_final[col] = ""

    df_final = df_final[cols_finales]
    df_final.to_csv(RUTA_SALIDA, index=False, encoding='utf-8-sig')
    
    print(f"\n✨ ¡ÉXITO! Guardado en: {RUTA_SALIDA}")
    # Ahora display() funcionará porque los tipos de datos están limpios
    display(df_final)
else:
    print("\n❌ No se encontraron datos.")

## ORDENES CONSOLIDADO

In [0]:
%pip install openpyxl xlrd pyxlsb

import pandas as pd
import os
import glob
import re

# ==========================================
# CONFIGURACIÓN PARA DATABRICKS
# ==========================================
RUTA_REPO = "/Workspace/Users/dilan.flores.h@uni.pe/CORPAC"
RUTA_BASE = f"{RUTA_REPO}/data_corpac/ordenes"
RUTA_SALIDA = f"{RUTA_REPO}/data_corpac/ordenes_consolidadas.csv"

def extraer_periodo_y_origen(nombre_archivo):
    n = nombre_archivo.upper()
    anio_match = re.search(r'20\d{2}', n)
    anio = anio_match.group(0) if anio_match else "S/A"
    
    origen = "SEDE CENTRAL"
    if "CUSCO" in n: origen = "CUSCO"
    elif "PROV" in n: origen = "PROVINCIAS"

    meses = {
        'ENERO': '01', 'FEBRERO': '02', 'MARZO': '03', 'ABRIL': '04', 'MAYO': '05', 'JUNIO': '06',
        'JULIO': '07', 'AGOSTO': '08', 'SETIEMBRE': '09', 'OCTUBRE': '10', 'NOVIEMBRE': '11', 'DICIEMBRE': '12'
    }
    for mes_nom, mes_num in meses.items():
        if mes_nom in n: return anio, f"MES {mes_num}", origen

    if any(x in n for x in ["IVTRIM", "4TRIM"]): p = "TRIM 4"
    elif any(x in n for x in ["IIITRIM", "3TRIM"]): p = "TRIM 3"
    elif any(x in n for x in ["IITRIM", "2TRIM"]): p = "TRIM 2"
    elif any(x in n for x in ["ITRIM", "1TRIM"]): p = "TRIM 1"
    else: p = "S/P"
    
    return anio, p, origen

def clean_currency(x):
    if pd.isna(x) or str(x).strip() == '': return 0.0
    if isinstance(x, (int, float)): return float(x)
    s = re.sub(r'[^\d.]', '', str(x).replace(',', '.'))
    try: return float(s)
    except: return 0.0

def clean_text_id(x):
    """Limpia RUCs y Números de Orden para evitar errores de visualización"""
    if pd.isna(x): return ""
    s = str(x).strip()
    if s.endswith('.0'): s = s[:-2]
    return s

def procesar_excel_ordenes(ruta, anio, periodo, origen):
    try:
        xl = pd.ExcelFile(ruta)
        dfs_de_este_archivo = []

        hojas_interes = [s for s in xl.sheet_names if any(x in s.upper() for x in ["F8", "ORDEN", "TRIM", "OC", "OS", "F18"])]
        if not hojas_interes: hojas_interes = [xl.sheet_names[0]]

        for nombre_hoja in hojas_interes:
            df_temp = pd.read_excel(ruta, sheet_name=nombre_hoja, nrows=30, header=None)
            
            fila_cabecera = None
            for i, row in df_temp.iterrows():
                valores_fila = [str(x).upper() for x in row.fillna('').values]
                if any("ORDEN" in s or "EMPRESA" in s for s in valores_fila) and any("PROVEEDOR" in s or "MONTO" in s for s in valores_fila):
                    fila_cabecera = i
                    break
            
            if fila_cabecera is not None:
                df = pd.read_excel(ruta, sheet_name=nombre_hoja, skiprows=fila_cabecera)
                df.columns = [re.sub(r'\s+', ' ', str(c)).strip().upper() for c in df.columns]

                mapeo = {
                    'NRO_ORDEN': ['NRO. DE LA ORDEN DE COMPRA O SERVICIO', 'NRO. DE LA ORDEN', 'NRO DE ORDEN'],
                    'LUGAR': ['LUGAR DE COMPRA O PRESTACIÓN DE SERVICIOS', 'LUGAR'],
                    'DESCRIPCION': ['DESCRIPCIÓN DE LA CONTRATACIÓN', 'DESCRIPCIÓN DEL SERVICIO PRESTADO', 'SÍNTESIS DE ESPECIFICACIONES TÉCNICAS'],
                    'RUC': ['RUC DEL PROVEEDOR O CONTRATISTA', 'RUC'],
                    'PROVEEDOR': ['NOMBRE DEL PROVEEDOR O CONTRATISTA', 'NOMBRE DEL PROVEEDOR', 'PROVEEDOR', 'NOMBRE DE LA EMPRESA'],
                    'MONTO_SOLES': ['MONTO DE LA ORDEN S/.', 'MONTO DE LA ORDEN S/', 'MONTO DE LA ORDEN', 'MONTO TOTAL CONTRATADO (S/.)', 'VALOR ESTIMADO(S/.']
                }

                dict_renombrar = {}
                for destino, origenes in mapeo.items():
                    for orig in origenes:
                        col_encontrada = next((c for c in df.columns if c == orig or c.startswith(f"{orig}_")), None)
                        if col_encontrada:
                            dict_renombrar[col_encontrada] = destino
                            break
                
                df = df.rename(columns=dict_renombrar)
                
                if 'MONTO_SOLES' in df.columns:
                    if isinstance(df['MONTO_SOLES'], pd.DataFrame): df['MONTO_SOLES'] = df['MONTO_SOLES'].iloc[:, 0]
                    df['MONTO_SOLES'] = df['MONTO_SOLES'].apply(clean_currency)
                
                # Limpieza de IDs críticos
                for col in ['RUC', 'NRO_ORDEN']:
                    if col in df.columns:
                        df[col] = df[col].apply(clean_text_id)

                df['ANIO'] = str(anio)
                df['PERIODO'] = periodo
                df['ORIGEN'] = origen
                df['ARCHIVO'] = os.path.basename(ruta)
                
                dfs_de_este_archivo.append(df)

        return pd.concat(dfs_de_este_archivo, ignore_index=True) if dfs_de_este_archivo else None

    except Exception as e:
        print(f"⚠️ Error en {os.path.basename(ruta)}: {e}")
        return None

# ==========================================
# PROCESO PRINCIPAL
# ==========================================
print(f"🚀 Iniciando consolidación de Órdenes en Databricks...")
listado_dfs = []

if os.path.exists(RUTA_BASE):
    archivos = glob.glob(os.path.join(RUTA_BASE, "**/*.xls*"), recursive=True)
    
    for f in archivos:
        if "~$" in f: continue
        nom = os.path.basename(f)
        anio_real, periodo, origen = extraer_periodo_y_origen(nom)
        
        if anio_real == "S/A":
            for p in f.split(os.sep):
                if p.isdigit() and len(p) == 4:
                    anio_real = p; break

        df_res = procesar_excel_ordenes(f, anio_real, periodo, origen)
        if df_res is not None:
            listado_dfs.append(df_res)
            print(f"✅ Procesado: {nom}")

if listado_dfs:
    df_final = pd.concat(listado_dfs, ignore_index=True, sort=False)
    
    # Lógica de Deduplicación
    def limpiar_nro_orden_comp(x):
        s = str(x).upper()
        s = re.sub(r'[A-Z\s\-]', '', s)
        try: return str(int(float(s)))
        except: return s

    if 'NRO_ORDEN' in df_final.columns:
        df_final['_NRO_LIMPIO'] = df_final['NRO_ORDEN'].apply(limpiar_nro_orden_comp)
        df_final = df_final.drop_duplicates(subset=['ANIO', 'MONTO_SOLES', '_NRO_LIMPIO'], keep='first')

    # Filtrar vacíos y forzar tipos string
    df_final = df_final[
        (df_final['PROVEEDOR'].notna()) & 
        (df_final['PROVEEDOR'].astype(str).str.strip() != '') & 
        (df_final['MONTO_SOLES'] > 0)
    ].copy()

    # Asegurar que las columnas de identificación sean strings para el display()
    for col in ['RUC', 'NRO_ORDEN', 'ANIO']:
        if col in df_final.columns:
            df_final[col] = df_final[col].astype(str)

    # Selección de columnas finales
    cols_finales = ['ANIO', 'PERIODO', 'ORIGEN', 'NRO_ORDEN', 'PROVEEDOR', 'RUC', 'MONTO_SOLES', 'ARCHIVO']
    df_final = df_final[[c for c in cols_finales if c in df_final.columns]]
    
    df_final.to_csv(RUTA_SALIDA, index=False, encoding='utf-8-sig')
    print(f"\n✨ ¡ÉXITO! Consolidado guardado en: {RUTA_SALIDA}")
    display(df_final)
else:
    print("\n❌ No se encontraron datos para procesar.")

## Hola Curay